## Earthquake-perturbed output signal visualisation
`demo_end_to_end(_cuda).py` saves earthquake-perturbed optical fibre output signals in Jones space.
This notebook visualises the output state of polarisation over time as a spectrogram, as in [\[4\]](#4) and [\[5\]](#5):

<a name="4">\[4\]</a>
R. M. Butler, J. Núñez-Kasaneva, *et al.*,
"End-to-End Modelling of Earthquake-Induced Polarisation Perturbations in Submarine Optical Fibres,"
In Review.

<a name="5">\[5\]</a>
Z. Zhan, M. Cantono *et al.*,
"Optical polarization-based seismic and water wave sensing on transoceanic cables,"
*Sci.*,
vol. 371, no. 6532, pp. 931&ndash;936,
Feb. 2021.
DOI: [10.1126/science.abe6648](https://doi.org/10.1126/science.abe6648)

We start by importing the necessary packages:

In [ ]:
from configparser import ConfigParser
import os
import logging

from tremor_waveplate_toolbox import Signal

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

logging.basicConfig(level = logging.DEBUG, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger()

Now, we define parameters to be used later in this notebook:

In [ ]:
# Paths to the configurations used by demo_end_to_end.py or demo_end_to_end_cuda.py
config_paths = [
        'config/earthquake_oaxaca.ini',
        'config/fibre_curie.ini',
        'config/signal_continuous.ini',
        'config/transceiver_curie.ini'
    ]

# Path to the saved data you want to visualise
fibre_output_path = 'results/propagated_signal_alpha=1.5.npy'

# System signal to noise ratio, estimated from Zhan et al.'s data from 0 Hz to 1 Hz
SNR = 277.49807780728

# Filtering parameters
averaging_window_duration = 200 # in seconds

# Spectrogram parameters
window_length = 2048
window = sp.signal.windows.blackmanharris(window_length)
hop = int(window_length / 4)
fourier_transform_lenght = int(window_length * 4)

# Visualisation parameters
plot_realisation = 0
minimum_spectrogram_value_dB = -4.5
maximum_spectrogram_value_dB = 0

# Saving parameters
png_path = 'results/propagated_signal_alpha=1.5_spectrogram.png'

Next, we load the configuration files that were used by `demo_end_to_end(_cuda).py` to generate fibre outputs:

In [ ]:
parameters = ConfigParser(inline_comment_prefixes = '#')
parameters.read(config_paths)

Next, load the fibre output saved by `demo_end_to_end(_cuda).py`:

In [ ]:
# The output data have shape [R, B, T, P] with realisation count R, batch size B, time sample count T, and polarisations P.
# end_to_end(_cuda).py varies earthquake perturbations in B (the earthquake is kept constant in T).
# In other words, T is the time axis of the signal (for a constant earthquake perturbation), and B is the time axis of the varying perturbation.
# We change the shape to [R, 1, B, P].
# Now, B is the time dimension for the rest of the notebook.
fibre_output = Signal(
        samples = np.load(fibre_output_path)[:, None, :, 0, :],
        sample_rate = parameters.getfloat('TRANSCEIVER', 'symbol_rate') * parameters.getint('TRANSCEIVER', 'sample_factor'),
        carrier_wavelength = parameters.getfloat('SIGNAL', 'carrier')
    )

The data from Zhan et al. had noise that is not modelled in `demo_end_to_end(_cuda).py`.
Now, we add this noise, based on an end-to-end system SNR estimated from Zhan et al.'s data from `0Hz` to `1Hz`.

In [ ]:
noise_power = fibre_output.power_W / SNR
noise = (np.random.default_rng().normal(size = fibre_output.shape) + 1j * np.random.default_rng().normal(size = fibre_output.shape)) * np.sqrt(noise_power[:, :, None, None] / 2 / 2) # Divide over 2 phases and 2 polarisations
fibre_output.samples_time += noise

Next, we go from Jones to Stokes space:

In [ ]:
samples_stokes = np.zeros(shape = (*fibre_output.shape[:-1], 4), dtype = float)

samples_stokes[..., 0] = np.abs(fibre_output.samples_time[..., 0]) ** 2 + np.abs(fibre_output.samples_time[..., 1]) ** 2
samples_stokes[..., 1] = np.abs(fibre_output.samples_time[..., 0]) ** 2 - np.abs(fibre_output.samples_time[..., 1]) ** 2
samples_stokes[..., 2] =  2 * np.real(fibre_output.samples_time[..., 0] * np.conj(fibre_output.samples_time[..., 1]))
samples_stokes[..., 3] = -2 * np.imag(fibre_output.samples_time[..., 0] * np.conj(fibre_output.samples_time[..., 1]))

samples_stokes[..., 1:] /= samples_stokes[..., 0, None]

fibre_output_stokes = Signal(
        samples = samples_stokes[..., 1:],
        sample_rate = fibre_output.sample_rate,
        carrier_wavelength = fibre_output.carrier_wavelength
    )

We subtract a moving average to rotate the state of polarisation to the north pole of the Poincaré sphere, as in [\[5\]](#5).
This essentially acts as a high-pass filter.

In [ ]:
averaging_window_sample_count = int(fibre_output_stokes[0].sample_rate * averaging_window_duration)
averaging_window = np.ones(shape = averaging_window_sample_count, dtype = complex)

fibre_output_stokes_moving_average = np.zeros(shape = fibre_output_stokes.shape[2:], dtype = float)
fibre_output_stokes_averaged = fibre_output_stokes.copy()
for realisation in fibre_output_stokes_averaged.samples_time:
    for batch in realisation:
        fibre_output_stokes_moving_average[:, 0] = np.convolve(batch[:, 0], averaging_window, 'same')
        fibre_output_stokes_moving_average[:, 1] = np.convolve(batch[:, 1], averaging_window, 'same')
        fibre_output_stokes_moving_average[:, 2] = np.convolve(batch[:, 2], averaging_window, 'same')

        # Calculate Stokes vector angles
        fibre_output_stokes_moving_average_yaw   = np.arctan2(fibre_output_stokes_moving_average[:, 1], fibre_output_stokes_moving_average[:, 0])
        fibre_output_stokes_moving_average_pitch = np.arctan2(fibre_output_stokes_moving_average[:, 2], np.linalg.norm(fibre_output_stokes_moving_average[:, :2], axis = 1))

        # Align stokes vector with S2 = 0 first, by setting the average yaw to 0.
        batch[:, :2] = np.einsum(
            'qps,sp->sq',
            np.array([
                [np.cos(-fibre_output_stokes_moving_average_yaw), -np.sin(-fibre_output_stokes_moving_average_yaw)],
                [np.sin(-fibre_output_stokes_moving_average_yaw),  np.cos(-fibre_output_stokes_moving_average_yaw)]
            ]),
            batch[:, :2]
        )

        # Align stokes vector with S1 = 0 next, by setting the average pitch to 0.
        batch[:, (0, 2)] = np.einsum(
            'qps,sp->sq',
            np.array([ # Compensate pitch to align with S1 axis, then add pi / 2 to align with S3 axis
                [np.cos(np.pi / 2 - fibre_output_stokes_moving_average_pitch), -np.sin(np.pi / 2 - fibre_output_stokes_moving_average_pitch)],
                [np.sin(np.pi / 2 - fibre_output_stokes_moving_average_pitch),  np.cos(np.pi / 2 - fibre_output_stokes_moving_average_pitch)]
            ]),
            batch[:, (0, 2)]
        )

Now, we calculate the spectrograms from Stokes parameters `S1` and `S2`, and add them.

In [ ]:
window_duration = window_length * fibre_output_stokes_averaged.sample_time
stfft = sp.signal.ShortTimeFFT(
        win = window,
        hop = hop,
        fs = fibre_output_stokes_averaged.sample_rate,
        mfft = fourier_transform_length,
        scale_to = 'psd'
    )

fibre_output_spectrograms = np.array([stfft.spectrogram(realisation.real[0, :, 0]) + stfft.spectrogram(realisation.real[0, :, 0]) for realisation in fibre_output_stokes_averaged.samples_time])
fibre_output_extents = [list(stfft.extent(realisation.sample_count)) for realisation in fibre_output_stokes_averaged]

Finally, plot the spectrogram in decibels:

In [ ]:
fig, ax = plt.subplots(figsize = (4.5, 4))

ax.imshow(
        np.log10(fibre_output_spectrograms[plot_realisation]),
        cmap = 'jet',
        vmin = minimum_spectrogram_value_dB,
        vmax = maximum_spectrogram_value_dB,
        aspect = 'auto',
        origin = 'lower',
        extent = fibre_output_extents[plot_realisation],
        interpolation = 'none'
    )
ax.set_ylabel(f"Frequency [Hz]")
ax.set_xlabel("Time [s]")
ax.set_ylim([1, 0])
ax.set_xlim([window_duration / 2, signal_stokes.duration - window_duration / 2])

If desired, save the spectrogram as a PNG:

In [ ]:
plt.imsave(
        png_path,
        np.log10(fibre_output_spectrograms[plot_realisation]),
        cmap = 'jet',
        vmin = minimum_spectrogram_value_dB,
        vmax = maximum_spectrogram_value_dB
    )